**Mini-Project: MCP + Agents AI integration in Gemini:**

In [28]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "mcp-server-git" \
  "fastmcp>=2.0.0"

In [29]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

print("API key loaded successfully")

API key loaded successfully


In [30]:
!node --version
!npx --version

v20.19.0
10.8.2


In [31]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Say hello in one short sentence.")

print(response.content)

Hello!


In [32]:
from pathlib import Path

WORKDIR = "/content/mcp_research_assistant"

Path(WORKDIR).mkdir(parents=True, exist_ok=True)

print("WORKDIR:", WORKDIR)

WORKDIR: /content/mcp_research_assistant


In [33]:
Path(WORKDIR, "article_1.txt").write_text(
"""
Artificial intelligence is increasingly used in education, healthcare, and finance.

AI systems can improve efficiency and decision making.

However, transparency, fairness, and human oversight remain important concerns.
""",
encoding="utf-8"
)

Path(WORKDIR, "article_2.txt").write_text(
"""
Model Context Protocol (MCP) allows AI systems to communicate with external tools.

MCP servers expose tools, resources, and prompts through a standardized interface.

This makes agent applications modular and easier to extend.
""",
encoding="utf-8"
)

print("Files created.")

Files created.


In [34]:
!ls -la /content/mcp_research_assistant

total 20
drwxr-xr-x 3 root root 4096 May 27 11:14 .
drwxr-xr-x 1 root root 4096 May 27 11:59 ..
-rw-r--r-- 1 root root  222 May 27 12:46 article_1.txt
-rw-r--r-- 1 root root  229 May 27 12:46 article_2.txt
drwxr-xr-x 8 root root 4096 May 27 11:31 .git


In [35]:
!cd /content/mcp_research_assistant && git init

Reinitialized existing Git repository in /content/mcp_research_assistant/.git/


In [36]:
!cd /content/mcp_research_assistant && git add .

In [37]:
!cd /content/mcp_research_assistant && git commit -m "Initial research files"

On branch master
nothing to commit, working tree clean


In [38]:
!cd /content/mcp_research_assistant && git config user.name "sergey-berezovka"
!cd /content/mcp_research_assistant && git config user.email "sergey.berezovkat@gmail.com"

In [39]:
!cd /content/mcp_research_assistant && git commit -m "Initial research files"

On branch master
nothing to commit, working tree clean


In [40]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
}

client = MultiServerMCPClient(
    mcp_connections,
    tool_name_prefix=True
)

print("MCP client created")

MCP client created


In [41]:
%%writefile /content/test_mcp_tools.py
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

WORKDIR = "/content/mcp_research_assistant"

async def main():
    mcp_connections = {
        "filesystem": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
        },
        "git": {
            "transport": "stdio",
            "command": "python",
            "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
        },
    }

    client = MultiServerMCPClient(
        mcp_connections,
        tool_name_prefix=True
    )

    tools = await client.get_tools()

    print("Tool count:", len(tools))
    for tool in tools:
        print(tool.name)

asyncio.run(main())

Overwriting /content/test_mcp_tools.py


In [42]:
!python /content/test_mcp_tools.py

Secure MCP Filesystem Server running on stdio
Client does not support MCP Roots, using allowed directories set from server args: [ '/content/mcp_research_assistant' ]
Tool count: 26
filesystem_read_file
filesystem_read_text_file
filesystem_read_media_file
filesystem_read_multiple_files
filesystem_write_file
filesystem_edit_file
filesystem_create_directory
filesystem_list_directory
filesystem_list_directory_with_sizes
filesystem_directory_tree
filesystem_move_file
filesystem_search_files
filesystem_get_file_info
filesystem_list_allowed_directories
git_git_status
git_git_diff_unstaged
git_git_diff_staged
git_git_diff
git_git_commit
git_git_add
git_git_reset
git_git_log
git_git_create_branch
git_git_checkout
git_git_show
git_git_branch


In [43]:
%%writefile /content/mcp_agent.py
import asyncio
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

WORKDIR = "/content/mcp_research_assistant"

async def main():

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
    )

    mcp_connections = {
        "filesystem": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
        },
        "git": {
            "transport": "stdio",
            "command": "python",
            "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
        },
    }

    client = MultiServerMCPClient(
        mcp_connections,
        tool_name_prefix=True
    )

    tools = await client.get_tools()

    agent = create_react_agent(llm, tools)

    result = await agent.ainvoke({
        "messages": [
            (
                "user",
                f"""
Use only files inside:

{WORKDIR}

1. List files in the project folder.
2. Read article_1.txt
3. Give a short summary.
"""
            )
        ]
    })

    print("\n===== AGENT MESSAGES =====\n")

    for i, msg in enumerate(result["messages"]):
        print(f"\n----- MESSAGE {i} -----\n")
        print(type(msg).__name__)
        print(msg.content)

asyncio.run(main())

Overwriting /content/mcp_agent.py


In [44]:
!python /content/mcp_agent.py

Secure MCP Filesystem Server running on stdio
Client does not support MCP Roots, using allowed directories set from server args: [ '/content/mcp_research_assistant' ]
/content/mcp_agent.py:35: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ig

In [45]:
%pip install -qU "fastmcp>=2.0.0"

In [63]:
from pathlib import Path
import textwrap

server_path = Path("/content/custom_mcp_server.py")

server_path.write_text(textwrap.dedent("""
from mcp.server.fastmcp import FastMCP
from typing import Dict, List

mcp = FastMCP("custom_ops")

@mcp.tool()
def ping() -> str:
    \"\"\"Health check tool.\"\"\"
    return "pong"

@mcp.tool()
def summarize_lines(lines: List[str]) -> Dict[str, int]:
    \"\"\"Return counts about a list of lines.\"\"\"
    total = len(lines)
    nonempty = sum(1 for l in lines if l.strip())
    return {"total_lines": total, "nonempty_lines": nonempty}

if __name__ == "__main__":
    mcp.run(transport="stdio")
"""), encoding="utf-8")

print("Wrote:", server_path)

Wrote: /content/custom_mcp_server.py


In [68]:
%%writefile /content/custom_mcp_server.py
from mcp.server.fastmcp import FastMCP
from typing import Dict, List
import logging

logging.disable(logging.CRITICAL)

mcp = FastMCP("custom_ops")

@mcp.tool()
def ping() -> str:
    return "pong"

@mcp.tool()
def summarize_lines(lines: List[str]) -> Dict[str, int]:
    total = len(lines)
    nonempty = sum(1 for l in lines if l.strip())

    return {
        "total_lines": total,
        "nonempty_lines": nonempty
    }

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting /content/custom_mcp_server.py


In [70]:
%%writefile /content/check_custom_mcp_tools.py
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {}

mcp_connections["custom_ops"] = {
    "transport": "stdio",
    "command": "python",
    "args": ["/content/custom_mcp_server.py"],
}

async def main():

    client2 = MultiServerMCPClient(
        mcp_connections,
        tool_name_prefix=True
    )

    tools2 = await client2.get_tools()

    print("Tool count:", len(tools2))
    print([t.name for t in tools2])

asyncio.run(main())

Writing /content/check_custom_mcp_tools.py


In [71]:
!python /content/check_custom_mcp_tools.py

Tool count: 2
['custom_ops_ping', 'custom_ops_summarize_lines']


In [72]:
%%writefile /content/check_all_mcp_tools.py
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

WORKDIR = "/content/mcp_research_assistant"
server_path = "/content/custom_mcp_server.py"

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [server_path],
    },
}

async def main():
    client = MultiServerMCPClient(
        mcp_connections,
        tool_name_prefix=True
    )

    tools = await client.get_tools()

    print("Tool count:", len(tools))
    print()

    for tool in tools:
        print(tool.name)

asyncio.run(main())

Writing /content/check_all_mcp_tools.py


In [73]:
!python /content/check_all_mcp_tools.py

Secure MCP Filesystem Server running on stdio
Client does not support MCP Roots, using allowed directories set from server args: [ '/content/mcp_research_assistant' ]
Tool count: 28

filesystem_read_file
filesystem_read_text_file
filesystem_read_media_file
filesystem_read_multiple_files
filesystem_write_file
filesystem_edit_file
filesystem_create_directory
filesystem_list_directory
filesystem_list_directory_with_sizes
filesystem_directory_tree
filesystem_move_file
filesystem_search_files
filesystem_get_file_info
filesystem_list_allowed_directories
git_git_status
git_git_diff_unstaged
git_git_diff_staged
git_git_diff
git_git_commit
git_git_add
git_git_reset
git_git_log
git_git_create_branch
git_git_checkout
git_git_show
git_git_branch
custom_ops_ping
custom_ops_summarize_lines
